# C: совместный геном системы и задающих токов

Полные Afferents, шаг 0.1 мс, без ПД/ПИД и ×10. Кубический геном: **60 полей, 56 свободных координат**; фиксированы a1, a2 и оба A=0.37. Квадратичный вариант: 58/54; табличный этап для пяти целей: 62/58.

Один контроллер обслуживает все цели. Коэффициенты токов загружаются вместе с ним. Токи зависят только от целевого угла, не от измеренного состояния. `GENOME_PATH` принимает совместный JSON или файл поколения; старые 52-полевые контроллеры также доступны. Обучение запускается только явно.


<!-- current-training -->
**Переобучение: времена мышц ≥50 мс.** Геном `joint_refined`. Завершено тестов 408/408, строгий успех 0/408. Перезапустите ядро. [Результаты](results/README.md).

**English.** Muscle times ≥50 ms; winner `joint_refined`. Completed 408/408; strict success 0/408. Restart the kernel. [Results](results/README.en.md).


<!-- mirror-genome-variants -->

Три сохранённые версии: [быстрые мышцы](best_genome_fast_muscle.json), [времена ≥50 мс](best_genome_slow_muscle.json), [времена ≥50 мс и симметрия](best_genome_slow_muscle_mirrow.json). Выберите версию через `GENOME_VARIANT`; `GENOME_PATH` позволяет указать другой файл. По умолчанию выбран зеркальный геном. Повторный запуск ячейки обновляет старые загрузчики в памяти ядра. [Происхождение и параметры](GENOME_VERSIONS.md).

Three saved variants: fast muscles, ≥50 ms muscles, and ≥50 ms with exact mirror symmetry. Select a named version using `GENOME_VARIANT`, or override it with `GENOME_PATH`. The mirror genome is selected by default. Rerunning the cell refreshes stale cached codecs. [Sources and values](GENOME_VERSIONS.en.md).

In [ ]:
from pathlib import Path
import os
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/SpikingNN/core/Izh_net.py").is_file())
sys.path.insert(0, str(REPO))
import importlib
from scripts.AfferentBalance2026.experiments import notebook_runtime as _notebook_runtime
# Refresh this helper too: an open kernel may retain its previous implementation.
importlib.reload(_notebook_runtime)
# Update core.genome before importing dependants that require MIRROR_INDICES.
_notebook_runtime.refresh_model_modules()
resolve_genome_path = _notebook_runtime.resolve_genome_path
from scripts.AfferentBalance2026.experiments.core.genome import GenomeConfig
from scripts.AfferentBalance2026.experiments.simulation import case_grid, simulate
from scripts.AfferentBalance2026.experiments.simulation import MODEL
if MODEL != "SpikingNN.Afferents/v3-tensile":
    raise RuntimeError("Restart the Jupyter kernel: tensile-torque model v3 is required.")
from scripts.AfferentBalance2026.experiments.visualization import (
    configure_fonts, load_solution, replay_grid, parameter_scan,
    plot_phase, plot_angles, plot_states,
)

EXPERIMENT = REPO / "scripts/AfferentBalance2026/experiments/C_setpoint_balance"
config = GenomeConfig(EXPERIMENT / "genome_cfg.json")
from scripts.AfferentBalance2026.experiments.C_setpoint_balance.joint_study import (
    SCHEMA as JOINT_SCHEMA, load_joint_solution, run_pipeline, sweep_joint,
)
# Choose fast_muscle, slow_muscle, slow_muscle_mirrow, or current.
GENOME_VARIANT = "slow_muscle_mirrow"
GENOME_PATH = None  # Optional explicit JSON/population path; overrides the variant.
USE_INITIAL_CONTROLLER = False  # True selects the untrained live configuration.
GENOME_PATH = (None if USE_INITIAL_CONTROLLER else
               resolve_genome_path(EXPERIMENT, GENOME_PATH, variant=GENOME_VARIANT))
RANK = 1
joint_genome = command = None
if GENOME_PATH is None:
    simulation_config, physical = config, config.initial_physical.copy()
    selection = {"selection": "initial controller (untrained)"}
else:
    path = Path(GENOME_PATH)
    if path.suffix == ".json":
        schema = json.loads(path.read_text()).get("schema")
    else:
        with np.load(path, allow_pickle=False) as data:
            schema = str(data["schema"])
    if schema == JOINT_SCHEMA:
        joint_genome, joint_unit, selection = load_joint_solution(path, rank=RANK)
        simulation_config = joint_genome.controller
        physical, command = joint_genome.decode(joint_unit)
    else:
        simulation_config, physical, selection = load_solution(path, rank=RANK)

# This environment variable is used only by automated notebook smoke checks.
SMOKE_TEST = os.environ.get("SPIKINGNN_NOTEBOOK_SMOKE") == "1"
DURATION_S = 0.02 if SMOKE_TEST else simulation_config.data["simulation"]["duration_s"]
TARGET_DEG = 0.0
CURRENTS = np.array([0.0, 0.0])  # Fallback only for historical controller-only genomes.
configure_fonts(13)
print(selection)
print("Fixed a1/a2 (m):", physical[46:48], "integration step: 0.1 ms")


In [ ]:
from scripts.AfferentBalance2026.experiments.C_setpoint_balance.polynomial_study import (
    load_current_law, polynomial_currents,
)
CURRENT_LAW_PATH = None  # Optional override for controller-only legacy solutions.
if joint_genome is not None and CURRENT_LAW_PATH is not None:
    raise ValueError("The joint genome already contains its command function")
coefficients = None if CURRENT_LAW_PATH is None else load_current_law(CURRENT_LAW_PATH)


def command_for(target_deg):
    """Evaluate the saved command using the desired target, never the measured state."""
    if joint_genome is not None:
        return joint_genome.currents(command, case_grid([0.], [0.], target_deg))[0]
    if coefficients is not None:
        return polynomial_currents(coefficients, np.deg2rad(target_deg))
    return CURRENTS.copy()


print("Target (deg):", TARGET_DEG, "currents:", command_for(TARGET_DEG))


## Функции для работы с выбранным геномом

`run_on_grid` рассчитывает сетку начальных условий; `view_states` показывает все основные переменные для одного примера. `scan_parameter` меняет один свободный параметр в допустимых пределах. Геометрия фиксирована: для её изменения создайте отдельный конфиг.

Фазовые графики сохраняют точки через 1 мс, но интегрирование и метрики всегда используют шаг 0.1 мс. Для подробного просмотра одной траектории записывается каждый шаг.

In [ ]:
def run_on_grid(angles_deg, speeds, target_deg=TARGET_DEG, currents=None):
    """Replay the selected genome on an absolute-angle initial-condition grid."""
    drive = command_for(target_deg) if currents is None else currents
    return replay_grid(simulation_config, physical, angles_deg, speeds,
                       target_deg=target_deg, currents=drive,
                       duration_s=DURATION_S, stride=10)


def view_states(theta0_deg=30.0, omega0=0.0):
    """Plot limb, neurons, muscles and full afferents for one initial condition."""
    record = replay_grid(simulation_config, physical, [theta0_deg], [omega0],
                         target_deg=TARGET_DEG, currents=command_for(TARGET_DEG),
                         duration_s=DURATION_S, stride=1)[0]
    for name, figure in plot_states(record).items():
        print(name)
        display(figure)
        plt.close(figure)
    return record


def phase_portrait(records):
    """Display genuine phase trajectories, including failed recovery attempts."""
    figure = plot_phase(records)
    display(figure)
    plt.close(figure)


def scan_parameter(name, values):
    """Compare one mutable genome parameter without changing the saved genome."""
    records = parameter_scan(simulation_config, physical, name, values,
                             target_deg=TARGET_DEG, currents=command_for(TARGET_DEG),
                             duration_s=DURATION_S)
    figure = plot_angles(records)
    display(figure)
    plt.close(figure)
    return records


## Фазовый портрет: сетка углов и ненулевых скоростей

По умолчанию 10×10: углы −50…50°, скорости −1…1 рад/с. Звезда обозначает цель, крестики — падения. Неуспешные траектории не исключаются.

In [ ]:
angles = np.linspace(-50, 50, 2 if SMOKE_TEST else 10)
speeds = np.linspace(-1, 1, 2 if SMOKE_TEST else 10)
phase_records = run_on_grid(angles, speeds)
phase_portrait(phase_records)
print("Successful cases:", sum(r["metrics"][5] for r in phase_records), "/", len(phase_records))


## Временные ряды угла при нулевой скорости

In [ ]:
angle_records = run_on_grid(angles, [0.0])
figure = plot_angles(angle_records)
display(figure)
plt.close(figure)


## Состояния одного примера

Напряжения и спайки нейронов, активации/силы/кальций мышц, Ia/II/Ib и движение конечности. Все сигналы относятся только к выбранному начальному условию.

In [ ]:
single_record = view_states(theta0_deg=-30.0, omega0=0.0)

## Сетка значений одного параметра генома

In [ ]:
field = simulation_config.data["parameters"][16]  # Flexor neuromuscular weight.
weights = np.geomspace(field["minimum"], field["maximum"], 3 if SMOKE_TEST else 5)
parameter_records = scan_parameter(field["name"], weights)


## Пять заданных углов

Для каждой цели вычисляются её собственные токи из одного сохранённого генома. Показаны реальные симуляции; этот блок не обучает параметры.


In [ ]:
PLOT_ALL_TARGETS = True
if PLOT_ALL_TARGETS:
    for target in config.data["setpoint"]["targets_deg"]:
        drive = command_for(target)
        records = run_on_grid(angles, speeds, target_deg=target, currents=drive)
        print("Target:", target, "currents:", drive)
        phase_portrait(records)


## Явный запуск обучения

`joint`: сеть + коэффициенты сразу; `staged`: общая сеть + таблица токов, МНК, совместное дообучение. 200 — общий бюджет эпох, в staged он делится между этапами. `RUN_STUDY=False` предотвращает случайный длительный расчёт.


In [ ]:
from datetime import datetime
RUN_STUDY = False
METHOD = "staged"  # "joint" or "staged"; both optimize the shared controller.
if RUN_STUDY:
    output = EXPERIMENT / "results" / (METHOD + "_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    seed_path = EXPERIMENT / config.data["setpoint"]["seed_genome"]
    result = run_pipeline(config, output, method=METHOD, degree=3, population=50,
                          generations=200, workers=8, probability=.1, seed=42,
                          seed_genome=seed_path)
    print(output, result)
